# 3 — High-gap + swear features

Inputs:
- `data/processed/1-imbault_gaps_va.csv`
- `data/processed/2-writestreak_posts_final_clean.csv`

Outputs:
- `data/processed/3-gap_and_swear_features.csv`


In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))
from l2affect.utils.config import load_config, resolve

cfg = load_config(REPO_ROOT / ".." / "configs" / "config.yaml")
processed_dir = resolve(cfg["paths"]["processed_dir"])


In [2]:
posts = pd.read_csv(processed_dir / "2-writestreak_posts_final_clean.csv")
gaps = pd.read_csv(processed_dir / "1-imbault_gaps_va.csv")

posts["tokens_list"] = posts["tokens"].map(lambda s: json.loads(s) if isinstance(s,str) else [])
gaps["word"] = gaps["word"].astype(str).str.strip().str.lower()
for c in ["gap_valence","gap_arousal"]:
    gaps[c] = pd.to_numeric(gaps[c], errors="coerce")
gaps = gaps.dropna(subset=["word","gap_valence","gap_arousal"]).copy()
gaps["gap_mag"] = np.sqrt(gaps["gap_valence"]**2 + gaps["gap_arousal"]**2) if "gap_mag" not in gaps.columns else pd.to_numeric(gaps["gap_mag"], errors="coerce")
gaps = gaps.dropna(subset=["gap_mag"]).copy()

gap_cfg = cfg.get("gap_definition", {})
mode = gap_cfg.get("mode", "quantile")
if mode == "absolute":
    thr = float(gap_cfg.get("absolute_threshold", 0.75))
else:
    q = float(gap_cfg.get("quantile", 0.90))
    thr = float(gaps["gap_mag"].quantile(q))

high = gaps[gaps["gap_mag"] >= thr].copy()
high_set = set(high["word"].tolist())

gap_mag = dict(zip(gaps["word"], gaps["gap_mag"]))
gap_val = dict(zip(gaps["word"], gaps["gap_valence"]))
gap_aro = dict(zip(gaps["word"], gaps["gap_arousal"]))

thr

1.838722112406205

In [3]:
SWEAR_WORDS = {
    "ass","asshole","bitch","boob","bullshit","butt","cock","damn","dickhead","dork","douche","dummy",
    "faggot","fuck","fucker","homo","horseshit","idiot","jackass","moron","piss","prick","pussy",
    "retard","screw","tit"
}
swear_set = {w.lower() for w in SWEAR_WORDS}

def compute_features(tokens):
    toks = [str(t).lower() for t in tokens]
    denom = max(1, len(toks))

    mags = [gap_mag[t] for t in toks if t in high_set]
    high_gap_count = len(mags)
    high_gap_density = high_gap_count / denom
    high_gap_gapmag_mean = float(np.mean(mags)) if mags else np.nan
    high_gap_gapmag_max = float(np.max(mags)) if mags else np.nan

    vals = [gap_val[t] for t in toks if t in high_set]
    aros = [gap_aro[t] for t in toks if t in high_set]
    high_gap_valence_mean_signed = float(np.mean(vals)) if vals else np.nan
    high_gap_arousal_mean_signed = float(np.mean(aros)) if aros else np.nan

    swear_hits = [t for t in toks if t in swear_set]
    swear_count = len(swear_hits)
    swear_density = swear_count / denom
    swear_present = 1 if swear_count > 0 else 0
    swear_unique_count = len(set(swear_hits))
    swear_types = json.dumps(sorted(set(swear_hits)))

    high_gap_swear_count = sum(1 for t in swear_hits if t in high_set)

    return (
        high_gap_count, high_gap_density, high_gap_gapmag_mean, high_gap_gapmag_max,
        high_gap_valence_mean_signed, high_gap_arousal_mean_signed,
        swear_count, swear_density, swear_present, swear_unique_count, swear_types,
        high_gap_swear_count
    )

feat_cols = [
    "high_gap_count","high_gap_density","high_gap_gapmag_mean","high_gap_gapmag_max",
    "high_gap_valence_mean_signed","high_gap_arousal_mean_signed",
    "swear_count","swear_density","swear_present","swear_unique_count","swear_types",
    "high_gap_swear_count"
]

feat_df = pd.DataFrame(posts["tokens_list"].map(compute_features).tolist(), columns=feat_cols, index=posts.index)
out = posts.drop(columns=["tokens_list"]).join(feat_df)
out.head()

,post_id,post_fullname,author,user_id,created_utc,created_at,title,selftext,subreddit,permalink,...,high_gap_gapmag_mean,high_gap_gapmag_max,high_gap_valence_mean_signed,high_gap_arousal_mean_signed,swear_count,swear_density,swear_present,swear_unique_count,swear_types,high_gap_swear_count
0,kt7xl7,t3_kt7xl7,Namatl,00052db50867a4da,1610129063,2021-01-08 18:04:23+00:00,Streak 1 Writing Prompt,"""I was walking to the store in the evening, th...",WriteStreakEN,/r/WriteStreakEN/comments/kt7xl7/streak_1_writ...,...,NaN,NaN,NaN,NaN,0,0.0,0,0,[],0
1,ku0tf4,t3_ku0tf4,Namatl,00052db50867a4da,1610230734,2021-01-09 22:18:54+00:00,Streak 2 Favorite Singer/Band,Mi favorite singer is Michael. I grew up liste...,WriteStreakEN,/r/WriteStreakEN/comments/ku0tf4/streak_2_favo...,...,NaN,NaN,NaN,NaN,0,0.0,0,0,[],0
2,j3vkx8,t3_j3vkx8,Stylelike,000a672e1864051f,1601649246,2020-10-02 14:34:06+00:00,Streak 1: My bike.,"In May, I bought my first motorcycle; my first...",WriteStreakEN,/r/WriteStreakEN/comments/j3vkx8/streak_1_my_b...,...,NaN,NaN,NaN,NaN,0,0.0,0,0,[],0
3,j4p7n0,t3_j4p7n0,Stylelike,000a672e1864051f,1601768545,2020-10-03 23:42:25+00:00,Streak 2: First aids,I think that learn first aids is indispensable...,WriteStreakEN,/r/WriteStreakEN/comments/j4p7n0/streak_2_firs...,...,2.326908,2.326908,-1.61,1.68,0,0.0,0,0,[],0
4,j5ct7h,t3_j5ct7h,Stylelike,000a672e1864051f,1601870579,2020-10-05 04:02:59+00:00,Streak 3: What would I do with 10 billion doll...,Well... if I can’t expend it in my family or m...,WriteStreakEN,/r/WriteStreakEN/comments/j5ct7h/streak_3_what...,...,NaN,NaN,NaN,NaN,0,0.0,0,0,[],0


In [7]:
out_path = processed_dir / "3-gap_and_swear_features.csv"
summary_path = processed_dir / "3-gap_and_swear_summary.csv"

out.to_csv(out_path, index=False)

summary = {
    "posts_n": int(len(out)),
    "users_n": int(out["user_id"].nunique()),
    "high_gap_threshold_gapmag": float(thr),
    "posts_with_any_high_gap": int((out["high_gap_count"] > 0).sum()),
    "pct_posts_with_any_high_gap": float((out["high_gap_count"] > 0).mean()),
    "posts_with_any_swear": int((out["swear_present"] == 1).sum()),
    "pct_posts_with_any_swear": float((out["swear_present"] == 1).mean()),
    "total_swear_tokens": int(out["swear_count"].sum()),
}

print("Wrote:", out_path)
summary

Wrote: C:\Users\hus44\Code\Directed-Reading-Project\data\processed\3-gap_and_swear_features.csv


{'posts_n': 1251,
 'users_n': 56,
 'high_gap_threshold_gapmag': 1.838722112406205,
 'posts_with_any_high_gap': 887,
 'pct_posts_with_any_high_gap': 0.7090327737809752,
 'posts_with_any_swear': 10,
 'pct_posts_with_any_swear': 0.007993605115907274,
 'total_swear_tokens': 11}